# Boston Habitability Survey by ZIP code

# Safety Score

In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.patches import Polygon, PathPatch
from matplotlib.collections import PatchCollection
from matplotlib.ticker import FuncFormatter
import urllib.request
from shapely.geometry import shape
from matplotlib.path import Path
import pgeocode


In [1]:
import pandas as pd
import numpy as np
import requests
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ================================================
# Boston Crime Data Analysis - Safety Score by ZIP
# ================================================

# Cell 2: 載入數據
def load_crime_data(file_path='boston_crime_data.csv'):
    """
    載入本地CSV檔案
    """
    print("載入犯罪數據...")
    
    # 讀取CSV
    df = pd.read_csv(file_path, low_memory=False)
    
    # 轉換日期格式
    df['OCCURRED_ON_DATE'] = pd.to_datetime(df['OCCURRED_ON_DATE'])
    
    # 只選擇2025年的數據
    df_2025 = df[df['OCCURRED_ON_DATE'].dt.year == 2025].copy()
    
    print(f"2025年犯罪記錄數: {len(df_2025)}")
    
    return df_2025

# Cell 3: ZIP Code對應函數
def get_boston_zips():
    """
    Boston ZIP codes與中心座標
    """
    boston_zips = {
        '02108': {'lat': 42.3577, 'lon': -71.0653, 'name': 'Beacon Hill'},
        '02109': {'lat': 42.3601, 'lon': -71.0539, 'name': 'Downtown'},
        '02110': {'lat': 42.3570, 'lon': -71.0515, 'name': 'Financial District'},
        '02111': {'lat': 42.3508, 'lon': -71.0595, 'name': 'Chinatown'},
        '02113': {'lat': 42.3655, 'lon': -71.0553, 'name': 'North End'},
        '02114': {'lat': 42.3613, 'lon': -71.0689, 'name': 'West End'},
        '02115': {'lat': 42.3426, 'lon': -71.1024, 'name': 'Fenway/Kenmore'},
        '02116': {'lat': 42.3497, 'lon': -71.0764, 'name': 'Back Bay'},
        '02118': {'lat': 42.3381, 'lon': -71.0726, 'name': 'South End'},
        '02119': {'lat': 42.3233, 'lon': -71.0849, 'name': 'Roxbury'},
        '02120': {'lat': 42.3329, 'lon': -71.0972, 'name': 'Mission Hill'},
        '02121': {'lat': 42.3070, 'lon': -71.0836, 'name': 'Dorchester'},
        '02122': {'lat': 42.2865, 'lon': -71.0582, 'name': 'Dorchester'},
        '02124': {'lat': 42.2816, 'lon': -71.0723, 'name': 'Dorchester'},
        '02125': {'lat': 42.3150, 'lon': -71.0589, 'name': 'Dorchester'},
        '02126': {'lat': 42.2730, 'lon': -71.0943, 'name': 'Mattapan'},
        '02127': {'lat': 42.3381, 'lon': -71.0389, 'name': 'South Boston'},
        '02128': {'lat': 42.3751, 'lon': -71.0391, 'name': 'East Boston'},
        '02129': {'lat': 42.3788, 'lon': -71.0612, 'name': 'Charlestown'},
        '02130': {'lat': 42.3100, 'lon': -71.1140, 'name': 'Jamaica Plain'},
        '02131': {'lat': 42.2845, 'lon': -71.1260, 'name': 'Roslindale'},
        '02132': {'lat': 42.2680, 'lon': -71.1586, 'name': 'West Roxbury'},
        '02134': {'lat': 42.3551, 'lon': -71.1320, 'name': 'Allston'},
        '02135': {'lat': 42.3478, 'lon': -71.1514, 'name': 'Brighton'},
        '02136': {'lat': 42.2550, 'lon': -71.1256, 'name': 'Hyde Park'},
    }
    return boston_zips

# Cell 4: 分配犯罪到ZIP Code
def assign_crime_to_zip(crime_df):
    """
    根據經緯度將犯罪分配到最近的ZIP
    """
    from math import radians, cos, sin, asin, sqrt
    
    def haversine(lon1, lat1, lon2, lat2):
        """計算兩點距離（英里）"""
        lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
        dlon = lon2 - lon1 
        dlat = lat2 - lat1 
        a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
        c = 2 * asin(sqrt(a)) 
        r = 3956
        return c * r
    
    boston_zips = get_boston_zips()
    
    print("將犯罪事件分配到ZIP codes...")
    
    def find_nearest_zip(lat, lon):
        if pd.isna(lat) or pd.isna(lon):
            return None
        
        try:
            lat = float(lat)
            lon = float(lon)
        except:
            return None
            
        min_dist = float('inf')
        nearest_zip = None
        
        for zip_code, info in boston_zips.items():
            dist = haversine(lon, lat, info['lon'], info['lat'])
            if dist < min_dist:
                min_dist = dist
                nearest_zip = zip_code
        
        # 只接受3英里內的犯罪
        if min_dist <= 3:
            return nearest_zip
        return None
    
    # 分配ZIP
    crime_df['ZIP_code'] = crime_df.apply(
        lambda row: find_nearest_zip(row['Lat'], row['Long']), 
        axis=1
    )
    
    # 過濾有效ZIP
    crime_with_zip = crime_df[crime_df['ZIP_code'].notna()].copy()
    print(f"成功分配 {len(crime_with_zip)} 筆犯罪到ZIP codes")
    
    return crime_with_zip

# Cell 5: 計算Safety Score
def calculate_safety_score(crime_with_zip):
    """
    計算每個ZIP的安全分數
    公式: count(INCIDENT_NUMBER) + shooting
    標準化: 0-10 (10=無犯罪, 0=6000起犯罪)
    """
    
    print("\n計算Safety Score...")
    
    # 處理SHOOTING欄位
    crime_with_zip['is_shooting'] = crime_with_zip['SHOOTING'].apply(
        lambda x: 1 if str(x).upper() in ['Y', 'YES', '1', 'TRUE'] else 0
    )
    
    # 按ZIP統計
    zip_stats = crime_with_zip.groupby('ZIP_code').agg({
        'INCIDENT_NUMBER': 'count',  # 事件總數
        'is_shooting': 'sum'         # 槍擊案數量
    }).reset_index()
    
    zip_stats.columns = ['ZIP_code', 'Incident_count', 'Shooting_count']
    
    # 計算危險分數
    zip_stats['Unsafe_Score'] = (
        zip_stats['Incident_count'] + 
        zip_stats['Shooting_count'] #* 0.2
    )
    
    # 標準化到0-10 (反向: 10=安全, 0=危險)
    # 根據要求: 6000起犯罪 = 0分, 0起犯罪 = 10分
    MAX_CRIME = 6000
    
    zip_stats['Safety_score'] = 10 * (1 - zip_stats['Unsafe_Score'] / MAX_CRIME)
    zip_stats['Safety_score'] = zip_stats['Safety_score'].clip(0, 10).round(2)
    
    # 確保所有Boston ZIP都包含
    all_zips = pd.DataFrame({'ZIP_code': list(get_boston_zips().keys())})
    final_result = all_zips.merge(zip_stats, on='ZIP_code', how='left')
    
    # 填充沒有犯罪的ZIP
    final_result['Incident_count'] = final_result['Incident_count'].fillna(0).astype(int)
    final_result['Shooting_count'] = final_result['Shooting_count'].fillna(0).astype(int)
    final_result['Unsafe_Score'] = final_result['Unsafe_Score'].fillna(0)
    final_result['Safety_score'] = final_result['Safety_score'].fillna(10)
    
    # 排序
    final_result = final_result.sort_values('ZIP_code')
    
    return final_result

# Cell 6: 主執行函數
def main():
    """
    主函數 - 執行完整分析
    """
    print("="*60)
    print("Boston Crime Analysis - Safety Score by ZIP")
    print("="*60)
    
    # 開檔案
    crime_df = load_crime_data('boston_crime_data.csv')
    
    # 分配犯罪到ZIP
    crime_with_zip = assign_crime_to_zip(crime_df)
    
    # 計算安全分數
    safety_scores = calculate_safety_score(crime_with_zip)
    
    # 輸出結果
    output_file = 'boston_safety_score.csv'
    safety_scores.to_csv(output_file, index=False)
    print(f"\n結果已儲存至: {output_file}")
    
    # 顯示統計
    print("\n" + "="*60)
    print("Safety Score統計摘要")
    print("="*60)
    
    print("\nTop 10 最安全的ZIP區域:")
    top10_safe = safety_scores.nlargest(10, 'Safety_score')[
        ['ZIP_code', 'Incident_count', 'Shooting_count', 'Safety_score']
    ]
    print(top10_safe.to_string(index=False))
    
    print("\n\nTop 10 最危險的ZIP區域:")
    top10_unsafe = safety_scores.nsmallest(10, 'Safety_score')[
        ['ZIP_code', 'Incident_count', 'Shooting_count', 'Safety_score']
    ]
    print(top10_unsafe.to_string(index=False))
    
    print("\n\n整體統計:")
    print(f"總犯罪事件: {safety_scores['Incident_count'].sum()}")
    print(f"總槍擊案: {safety_scores['Shooting_count'].sum()}")
    print(f"平均安全分數: {safety_scores['Safety_score'].mean():.2f}")
    
    return safety_scores

# Cell 7: 執行分析
if __name__ == "__main__":
    # 執行主程式
    results = main()
    
    # 顯示結果
    print("\n最終結果預覽:")
    print(results)

Boston Crime Analysis - Safety Score by ZIP
載入犯罪數據...
2025年犯罪記錄數: 74837
將犯罪事件分配到ZIP codes...
成功分配 70213 筆犯罪到ZIP codes

計算Safety Score...

結果已儲存至: boston_safety_score.csv

Safety Score統計摘要

Top 10 最安全的ZIP區域:
ZIP_code  Incident_count  Shooting_count  Safety_score
   02110             604               0          8.99
   02114             767               3          8.72
   02132             944               9          8.41
   02109            1016               0          8.31
   02129            1012               3          8.31
   02113            1266               0          7.89
   02115            1551               1          7.41
   02134            1664               8          7.21
   02127            1716               5          7.13
   02108            1765               2          7.06


Top 10 最危險的ZIP區域:
ZIP_code  Incident_count  Shooting_count  Safety_score
   02118            7594              22          0.00
   02121            5146             108          1.24
   

In [3]:
# 然後檢查特定ZIP
target_zips = ['02210', '02215']
specific_zips = results[results['ZIP_code'].isin(target_zips)]
print("ZIP 02210 和 02215 的資料：")
print(specific_zips)

# 查看是否有犯罪記錄
for _, row in specific_zips.iterrows():
    print(f"\nZIP {row['ZIP_code']}:")
    print(f"  犯罪事件: {row['Incident_count']}")
    print(f"  槍擊案: {row['Shooting_count']}")
    print(f"  安全分數: {row['Safety_score']}")

ZIP 02210 和 02215 的資料：
Empty DataFrame
Columns: [ZIP_code, Incident_count, Shooting_count, Unsafe_Score, Safety_score]
Index: []


In [4]:
# add data: 02210, 02215
new_rows = pd.DataFrame({
    'ZIP_code': ['02210', '02215'],
    'Incident_count': [0, 0],
    'Shooting_count': [0, 0],
    'Unsafe_Score': [0.0, 0.0],
    'Safety_score': [10.0, 10.0]
})

# merge into
results = pd.concat([results, new_rows], ignore_index=True)
# sort
results = results.sort_values('ZIP_code').reset_index(drop=True)
results.to_csv('boston_safety_score.csv', index=False)


# School

In [5]:
import pandas as pd
import numpy as np

# ================================================
# NCES學校數據解析器
# ================================================

df_school = pd.read_excel('boston_school_data.xlsx', sheet_name='sheet2')

low = df_school['Low Grade*']
high = df_school['High Grade*']

#### boston_school_level.xlsx
add column"Simple_Level": Elementary/ Middle/ High/ Other

In [6]:
import pandas as pd
import numpy as np

# 讀取數據
df_school = pd.read_excel('boston_school_data.xlsx', sheet_name='sheet2')

def classify_school_level(df_school):
    """
    根據Low Grade和High Grade判斷學校類型
    
    NCES Grade代碼說明：
    PK = Pre-Kindergarten (學前班)
    KG = Kindergarten (幼稚園)
    01-12 = 1年級到12年級
    13 = Post-secondary (大學以上)
    UG = Ungraded (無年級)
    """
    
    def get_school_type(row):
        low = row['Low Grade*']
        high = row['High Grade*']
        
        # 轉換grade代碼為數字
        grade_map = {
            'PK': -2, 'PreK': -2,
            'KG': 0, 'K': 0,
            '01': 1, '02': 2, '03': 3, '04': 4, '05': 5,
            '06': 6, '07': 7, '08': 8, '09': 9,
            '10': 10, '11': 11, '12': 12,
            '13': 13, 'UG': 99
        }
        
        # 處理字符串格式
        if pd.isna(low) or pd.isna(high):
            return 'Unknown'
        
        low_str = str(low).strip()
        high_str = str(high).strip()
        
        # 轉換為數字
        low_num = grade_map.get(low_str, int(low_str) if low_str.isdigit() else -1)
        high_num = grade_map.get(high_str, int(high_str) if high_str.isdigit() else -1)
        
        # 分類邏輯
        if low_num <= 0 and high_num <= 0:
            return 'Pre-School'  # 學前教育
        elif low_num <= 0 and high_num <= 5:
            return 'Elementary'  # 小學
        elif low_num <= 0 and high_num == 8:
            return 'K-8'  # K-8學校
        elif low_num <= 0 and high_num >= 12:
            return 'K-12'  # K-12學校
        elif 6 <= low_num <= 8 and high_num <= 8:
            return 'Middle'  # 中學
        elif low_num >= 9 and high_num == 12:
            return 'High'  # 高中
        elif low_num >= 6 and high_num == 12:
            return 'Secondary'  # 中學+高中
        elif low_num == 13 or high_num == 13:
            return 'Post-Secondary'  # 大學/職業學校
        else:
            return 'Other'
    
    # 應用分類
    df_school['School_Level'] = df_school.apply(get_school_type, axis=1)
    
    # 創建更簡單的分類
    def simple_level(level):
        if level in ['Elementary', 'K-8', 'K-12']:
            return 'Elementary'
        elif level in ['Middle', 'Secondary']:
            return 'Middle'
        elif level == 'High':
            return 'High'
        elif level == 'Pre-School':
            return 'Pre-K'
        else:
            return 'Other'
    
    df_school['Simple_Level'] = df_school['School_Level'].apply(simple_level)
    
    return df_school

# 執行分類
df_school = classify_school_level(df_school)

# ================================================
# Sheet 1: Overall - Group by ZIP and Simple_Level
# ================================================

# Group by ZIP and Simple_Level，計算個數
df_overall = df_school.groupby(['ZIP', 'Simple_Level']).size().reset_index(name='Count')

# 按ZIP和Simple_Level排序
df_overall = df_overall.sort_values(['ZIP', 'Simple_Level'])

print("="*60)
print("Sheet 1: Overall (Group by ZIP and Simple_Level)")
print("="*60)
print(df_overall.head(10))
print(f"\n總共 {len(df_overall)} 個ZIP-Level組合")

# ================================================
# Sheet 2: 原始數據（已添加分類）
# ================================================

print("\n" + "="*60)
print("Sheet 2: 原始數據（含School_Level和Simple_Level）")
print("="*60)
print(f"總共 {len(df_school)} 所學校")
print("\n學校類型分布：")
print(df_school['Simple_Level'].value_counts())

# ================================================
# 輸出到Excel文件
# ================================================

# 創建Excel writer對象
output_file = 'boston_school_level.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Sheet 1: Overall
    df_overall.to_excel(writer, sheet_name='overall', index=False)
    
    # Sheet 2: 原始數據（含分類）
    df_school.to_excel(writer, sheet_name='detail', index=False)
    
    # 獲取workbook和worksheets
    workbook = writer.book
    
    # 調整overall sheet的列寬
    worksheet1 = writer.sheets['overall']
    worksheet1.column_dimensions['A'].width = 12  # ZIP
    worksheet1.column_dimensions['B'].width = 15  # Simple_Level
    worksheet1.column_dimensions['C'].width = 10  # Count

print(f"\n檔案已保存為: {output_file}")

# ================================================
# 額外分析：創建ZIP教育質量摘要
# ================================================

print("\n" + "="*60)
print("額外分析：ZIP教育質量摘要")
print("="*60)

# 創建pivot table來查看每個ZIP的學校分布
pivot_summary = df_school.pivot_table(
    index='ZIP',
    columns='Simple_Level',
    aggfunc='size',
    fill_value=0
).reset_index()

# 計算總學校數
pivot_summary['Total_Schools'] = pivot_summary.iloc[:, 1:].sum(axis=1)

# 顯示前10個ZIP的學校分布
print("\nTop 10 ZIPs by Total Schools:")
top_zips = pivot_summary.nlargest(10, 'Total_Schools')
print(top_zips)


Sheet 1: Overall (Group by ZIP and Simple_Level)
    ZIP Simple_Level  Count
0  2111   Elementary      1
1  2111       Middle      1
2  2113   Elementary      1
3  2115         High      1
4  2115       Middle      1
5  2116         High      2
6  2116        Other      1
7  2118   Elementary      1
8  2118        Other      1
9  2119   Elementary      7

總共 71 個ZIP-Level組合

Sheet 2: 原始數據（含School_Level和Simple_Level）
總共 154 所學校

學校類型分布：
Simple_Level
Other         56
Elementary    53
Middle        23
High          22
Name: count, dtype: int64

檔案已保存為: boston_school_level.xlsx

額外分析：ZIP教育質量摘要

Top 10 ZIPs by Total Schools:
Simple_Level   ZIP  Elementary  High  Middle  Other  Total_Schools
5             2119           7     3       1      4             15
13            2128           3     0       1      8             12
21            2150           1     3       1      7             12
22            2151           6     2       3      0             11
9             2124           5     1 

#### Boston_school_scores.csv
Score:
1. Elementary 0.8, Middle 0.6, High 0.6, Other 0.5
2. Total 0.2
3. Having all E,M,H +0.5 同時有E+M+H的ZIP額外加0.5分
4. normalize to 0-10

In [32]:
import pandas as pd
import numpy as np

def process_school_data(school_file='boston_school_level.xlsx'):
    """
    處理學校數據並計算0-10分的教育評分
    """
    
    print("="*60)
    print("Boston學校數據處理 - 教育評分計算")
    print("="*60)
    
    # 1. 讀取overall sheet（按ZIP和Simple_Level分組的數據）
    print("\n1. 讀取學校數據...")
    df_overall = pd.read_excel(school_file, sheet_name='overall')
    print(f"   讀取 {len(df_overall)} 筆資料")
    
    # 2. Pivot數據，將Simple_Level變成欄位
    print("\n2. 轉換數據格式...")
    school_pivot = df_overall.pivot_table(
        index='ZIP',
        columns='Simple_Level',
        values='Count',
        fill_value=0
    ).reset_index()
    
    # 重命名欄位
    school_pivot.columns.name = None
    column_mapping = {
        'Elementary': 'School_Elementary',
        'Middle': 'School_Middle',
        'High': 'School_High',
        'Other': 'School_Other',
        'Pre-K': 'School_PreK'
    }
    
    for old_name, new_name in column_mapping.items():
        if old_name in school_pivot.columns:
            school_pivot.rename(columns={old_name: new_name}, inplace=True)
    
    # 計算總學校數
    school_cols = [col for col in school_pivot.columns if col.startswith('School_')]
    school_pivot['Total_Schools'] = school_pivot[school_cols].sum(axis=1)
    
    print(f"   處理 {len(school_pivot)} 個ZIP區域")
    
    # 3. 計算教育評分（含完整性獎勵）
    print("\n3. 計算教育評分...")
    
    # 初始化分數
    raw_score = pd.Series(0.0, index=school_pivot.index)
    
    # 基礎分數計算
    if 'School_Elementary' in school_pivot.columns:
        raw_score += school_pivot['School_Elementary'] * 0.8
        print(f"   Elementary: {school_pivot['School_Elementary'].sum()} 所學校")
    
    if 'School_Middle' in school_pivot.columns:
        raw_score += school_pivot['School_Middle'] * 0.6
        print(f"   Middle: {school_pivot['School_Middle'].sum()} 所學校")
    
    if 'School_High' in school_pivot.columns:
        raw_score += school_pivot['School_High'] * 0.6
        print(f"   High: {school_pivot['School_High'].sum()} 所學校")
    
    if 'School_Other' in school_pivot.columns:
        raw_score += school_pivot['School_Other'] * 0.5
        print(f"   Other: {school_pivot['School_Other'].sum()} 所學校")
    
    # 總學校數獎勵
    raw_score += school_pivot['Total_Schools'] * 0.2
    
    # 完整性獎勵（同時有E+M+H的ZIP加0.5分）
    has_elementary = school_pivot.get('School_Elementary', 0) > 0
    has_middle = school_pivot.get('School_Middle', 0) > 0
    has_high = school_pivot.get('School_High', 0) > 0
    
    completeness_bonus = has_elementary & has_middle & has_high
    raw_score[completeness_bonus] += 0.5
    
    complete_zips = completeness_bonus.sum()
    print(f"\n   有完整K-12教育的ZIP數量: {complete_zips}")
    
    # 儲存原始分數
    school_pivot['Raw_Education_Score'] = raw_score.round(2)
    
    # 4. 標準化到0-20分
    print("\n4. 標準化分數到0-10...")
    max_score = raw_score.max()
    min_score = 0
    
    if max_score > 0:
        normalized_score = (raw_score / max_score) * 10
    else:
        normalized_score = pd.Series(0.0, index=school_pivot.index)
    
    school_pivot['Education_Score_10'] = normalized_score.clip(0, 10).round(2)
    
    print(f"   最高原始分: {max_score:.2f}")
    print(f"   最高標準分: 10.00")
    print(f"   最低標準分: 0.00")
    
    # 5. 加入額外資訊欄位
    school_pivot['Has_Complete_K12'] = completeness_bonus.map({True: 'Yes', False: 'No'})
    
    # 6. 重新排序欄位
    output_columns = [
        'ZIP',
        'School_Elementary',
        'School_Middle', 
        'School_High',
        'School_Other'
    ]
    
    # 加入PreK如果存在
    if 'School_PreK' in school_pivot.columns:
        output_columns.append('School_PreK')
    
    output_columns.extend([
        'Total_Schools',
        'Has_Complete_K12',
        'Raw_Education_Score',
        'Education_Score_10'
    ])
    
    # 確保所有欄位存在
    for col in output_columns:
        if col not in school_pivot.columns:
            if col.startswith('School_'):
                school_pivot[col] = 0
    
    final_df = school_pivot[output_columns].sort_values('ZIP')
    
    # 7. 輸出CSV
    output_file = 'Boston_school_scores.csv'
    final_df.to_csv(output_file, index=False)
    print(f"\n5. 檔案已儲存: {output_file}")
    
    # 8. 顯示統計摘要
    print("\n" + "="*60)
    print("教育評分統計摘要")
    print("="*60)
    
    print("\nTop 10 教育評分最高的ZIP:")
    top10 = final_df.nlargest(10, 'Education_Score_10')[
        ['ZIP', 'Total_Schools', 'Has_Complete_K12', 'Education_Score_10']
    ]
    print(top10.to_string(index=False))
    
    print("\n\n教育評分分布:")
    print(f"10-8分 (優秀): {len(final_df[final_df['Education_Score_10'] >= 16])} 個ZIP")
    print(f"8-6分  (良好): {len(final_df[(final_df['Education_Score_10'] >= 12) & (final_df['Education_Score_10'] < 16)])} 個ZIP")
    print(f"6-4分  (中等): {len(final_df[(final_df['Education_Score_10'] >= 8) & (final_df['Education_Score_10'] < 12)])} 個ZIP")
    print(f"4-2分  (較差): {len(final_df[(final_df['Education_Score_10'] >= 4) & (final_df['Education_Score_10'] < 8)])} 個ZIP")
    print(f"2-0分  (很差): {len(final_df[final_df['Education_Score_10'] < 4])} 個ZIP")
    
    return final_df

# 執行主程式
if __name__ == "__main__":
    # 處理數據並輸出CSV
    education_df = process_school_data('boston_school_level.xlsx')
    
    print("\n" + "="*60)
    print("處理完成！")
    print("輸出檔案: Boston_school_scores.csv")
    print("="*60)

Boston學校數據處理 - 教育評分計算

1. 讀取學校數據...
   讀取 71 筆資料

2. 轉換數據格式...
   處理 25 個ZIP區域

3. 計算教育評分...
   Elementary: 53 所學校
   Middle: 23 所學校
   High: 22 所學校
   Other: 56 所學校

   有完整K-12教育的ZIP數量: 9

4. 標準化分數到0-10...
   最高原始分: 13.50
   最高標準分: 10.00
   最低標準分: 0.00

5. 檔案已儲存: Boston_school_scores.csv

教育評分統計摘要

Top 10 教育評分最高的ZIP:
 ZIP  Total_Schools Has_Complete_K12  Education_Score_10
2119             15              Yes               10.00
2151             11              Yes                7.78
2150             12              Yes                7.11
2128             12               No                6.96
2124             10              Yes                6.81
2130              9              Yes                6.15
2136              8              Yes                4.96
2121              8               No                4.67
2125              8               No                4.67
2135              7              Yes                4.67


教育評分分布:
10-8分 (優秀): 0 個ZIP
8-6分  (良好): 0 個ZIP
6-4分 

# Public Transportation
Score:
1. 公車(路線+站牌)×0.6 + 火車(路線+站牌)×0.8 + 地鐵(路線+站牌)×0.8
2. 標準化到0-10

In [30]:
import pandas as pd
import numpy as np
from math import radians, cos, sin, asin, sqrt
import zipfile
import os

# ================================================
# MBTA交通分析 - 修正版
# 計算經過每個ZIP的公車和地鐵路線
# ================================================

def extract_gtfs_files(zip_path='MBTA_GTFS.zip'):
    """
    解壓GTFS檔案
    """
    print("解壓GTFS檔案...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('MBTA_data')
    print("解壓完成！")
    return 'MBTA_data'

def load_MBTA_data(folder_path='MBTA_data'):
    """
    讀取GTFS核心檔案
    """
    print("\n讀取GTFS數據...")
    
    # 讀取必要檔案
    stops = pd.read_csv(f'{folder_path}/stops.txt')
    routes = pd.read_csv(f'{folder_path}/routes.txt')
    stop_times = pd.read_csv(f'{folder_path}/stop_times.txt')
    trips = pd.read_csv(f'{folder_path}/trips.txt')
    
    print(f"- 站點數: {len(stops)}")
    print(f"- 路線數: {len(routes)}")
    print(f"- 班次數: {len(trips)}")
    
    return stops, routes, stop_times, trips

def get_boston_zips_boundaries():
    """
    Boston ZIP codes與邊界（使用更寬鬆的邊界判定）
    """
    # ZIP邊界定義（簡化的矩形邊界）
    boston_zips = {
        '02108': {'lat': 42.3577, 'lon': -71.0653, 'name': 'Beacon Hill', 'radius': 0.5},
        '02109': {'lat': 42.3601, 'lon': -71.0539, 'name': 'Downtown', 'radius': 0.5},
        '02110': {'lat': 42.3570, 'lon': -71.0515, 'name': 'Financial District', 'radius': 0.5},
        '02111': {'lat': 42.3508, 'lon': -71.0595, 'name': 'Chinatown', 'radius': 0.5},
        '02113': {'lat': 42.3655, 'lon': -71.0553, 'name': 'North End', 'radius': 0.5},
        '02114': {'lat': 42.3613, 'lon': -71.0689, 'name': 'West End', 'radius': 0.5},
        '02115': {'lat': 42.3426, 'lon': -71.1024, 'name': 'Fenway/Kenmore', 'radius': 0.8},
        '02116': {'lat': 42.3497, 'lon': -71.0764, 'name': 'Back Bay', 'radius': 0.6},
        '02118': {'lat': 42.3381, 'lon': -71.0726, 'name': 'South End', 'radius': 0.6},
        '02119': {'lat': 42.3233, 'lon': -71.0849, 'name': 'Roxbury', 'radius': 1.0},
        '02120': {'lat': 42.3329, 'lon': -71.0972, 'name': 'Mission Hill', 'radius': 0.7},
        '02121': {'lat': 42.3070, 'lon': -71.0836, 'name': 'Dorchester', 'radius': 1.0},
        '02122': {'lat': 42.2865, 'lon': -71.0582, 'name': 'Dorchester', 'radius': 1.0},
        '02124': {'lat': 42.2816, 'lon': -71.0723, 'name': 'Dorchester', 'radius': 1.2},
        '02125': {'lat': 42.3150, 'lon': -71.0589, 'name': 'Dorchester', 'radius': 0.8},
        '02126': {'lat': 42.2730, 'lon': -71.0943, 'name': 'Mattapan', 'radius': 1.0},
        '02127': {'lat': 42.3381, 'lon': -71.0389, 'name': 'South Boston', 'radius': 1.2},
        '02128': {'lat': 42.3751, 'lon': -71.0391, 'name': 'East Boston', 'radius': 1.2},
        '02129': {'lat': 42.3788, 'lon': -71.0612, 'name': 'Charlestown', 'radius': 0.8},
        '02130': {'lat': 42.3100, 'lon': -71.1140, 'name': 'Jamaica Plain', 'radius': 1.2},
        '02131': {'lat': 42.2845, 'lon': -71.1260, 'name': 'Roslindale', 'radius': 1.0},
        '02132': {'lat': 42.2680, 'lon': -71.1586, 'name': 'West Roxbury', 'radius': 1.5},
        '02134': {'lat': 42.3551, 'lon': -71.1320, 'name': 'Allston', 'radius': 1.0},
        '02135': {'lat': 42.3478, 'lon': -71.1514, 'name': 'Brighton', 'radius': 1.2},
        '02136': {'lat': 42.2550, 'lon': -71.1256, 'name': 'Hyde Park', 'radius': 1.2},
    }
    
    # 特別處理South Boston的Andrew站
    # Andrew站座標約: 42.3300, -71.0567
    
    return boston_zips

def haversine(lon1, lat1, lon2, lat2):
    """計算兩點間距離（英里）"""
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    r = 3956  # 地球半徑（英里）
    return c * r

def assign_stops_to_multiple_zips(stops_df, boston_zips):
    """
    將站點分配到ZIP（一個站點可能屬於多個ZIP）
    使用更寬鬆的範圍來捕捉邊界站點
    """
    print("\n將站點分配到ZIP codes...")
    
    # 為每個站點創建ZIP列表
    stop_zips = []
    
    for idx, stop in stops_df.iterrows():
        lat = stop['stop_lat']
        lon = stop['stop_lon']
        
        if pd.isna(lat) or pd.isna(lon):
            continue
        
        # 檢查每個ZIP
        for zip_code, info in boston_zips.items():
            dist = haversine(lon, lat, info['lon'], info['lat'])
            
            # 使用ZIP特定的半徑
            if dist <= info['radius']:
                stop_zips.append({
                    'stop_id': stop['stop_id'],
                    'stop_name': stop['stop_name'],
                    'ZIP': zip_code,
                    'distance': dist
                })
    
    # 轉換為DataFrame
    stop_zip_df = pd.DataFrame(stop_zips)
    
    print(f"找到 {len(stop_zip_df)} 個站點-ZIP配對")
    
    return stop_zip_df

def analyze_routes_by_zip(stop_zip_df, stops_df, routes_df, stop_times_df, trips_df):
    """
    分析經過每個ZIP的路線
    """
    print("\n分析路線類型...")
    
    # GTFS route_type代碼
    route_type_names = {
        0: 'Light Rail',  # 綠線輕軌
        1: 'Subway',      # 紅線、藍線、橙線地鐵
        2: 'Commuter Rail',
        3: 'Bus',
        4: 'Ferry'
    }
    
    routes_df['route_type_name'] = routes_df['route_type'].map(route_type_names)
    
    # 找出每個站點有哪些路線
    print("計算站點-路線關係...")
    
    # 從stop_times和trips找出站點對應的路線
    stop_trips = stop_times_df[['stop_id', 'trip_id']].drop_duplicates()
    stop_routes = stop_trips.merge(trips_df[['trip_id', 'route_id']], on='trip_id')
    stop_routes = stop_routes[['stop_id', 'route_id']].drop_duplicates()
    
    # 加入路線類型資訊
    stop_routes = stop_routes.merge(
        routes_df[['route_id', 'route_type', 'route_type_name', 'route_short_name', 'route_long_name']], 
        on='route_id'
    )
    
    # 合併到stop_zip_df
    zip_routes = stop_zip_df.merge(stop_routes, on='stop_id')
    
    return zip_routes

def calculate_transit_by_zip(zip_routes_df):
    """
    計算每個ZIP的交通統計
    按照要求的公式計算
    """
    print("\n計算ZIP層級統計...")
    
    # 計算每個ZIP的唯一站點數和路線數
    zip_stats = []
    
    for zip_code in zip_routes_df['ZIP'].unique():
        zip_data = zip_routes_df[zip_routes_df['ZIP'] == zip_code]
        
        # 計算唯一站點數（站牌數量）
        bus_stops = len(zip_data[zip_data['route_type'] == 3]['stop_id'].unique())
        subway_stops = len(zip_data[zip_data['route_type'] == 1]['stop_id'].unique())
        light_rail_stops = len(zip_data[zip_data['route_type'] == 0]['stop_id'].unique())
        commuter_rail_stops = len(zip_data[zip_data['route_type'] == 2]['stop_id'].unique())
        
        # 計算唯一路線數（經過的車次）
        bus_routes = len(zip_data[zip_data['route_type'] == 3]['route_id'].unique())
        subway_routes = len(zip_data[zip_data['route_type'] == 1]['route_id'].unique())
        light_rail_routes = len(zip_data[zip_data['route_type'] == 0]['route_id'].unique())
        commuter_rail_routes = len(zip_data[zip_data['route_type'] == 2]['route_id'].unique())
        
        # Train = Subway + Light Rail + Commuter Rail
        train_stops = subway_stops + light_rail_stops + commuter_rail_stops
        train_routes = subway_routes + light_rail_routes + commuter_rail_routes
        
        zip_stats.append({
            'ZIP': zip_code,
            'bus_stops': bus_stops,
            'bus_routes': bus_routes,
            'train_stops': train_stops,
            'train_routes': train_routes,
            'subway_stops': subway_stops,
            'subway_routes': subway_routes,
            'light_rail_stops': light_rail_stops,
            'light_rail_routes': light_rail_routes,
            'commuter_rail_stops': commuter_rail_stops,
            'commuter_rail_routes': commuter_rail_routes
        })
    
    # 轉換為DataFrame
    zip_summary = pd.DataFrame(zip_stats)
    
    # 按照指定公式計算分數
    # 公車(數量&站牌)*0.6 + train(數量&站牌)*0.8 + subway(數量&站牌)*0.8
    zip_summary['raw_score'] = (
        (zip_summary['bus_routes'] + zip_summary['bus_stops']) * 0.6 +
        (zip_summary['train_routes'] + zip_summary['train_stops']) * 0.8 +
        (zip_summary['subway_routes'] + zip_summary['subway_stops']) * 0.8
    )
    
    # Normalize到0-10分
    min_score = 0  # 最低分固定為0
    max_score = zip_summary['raw_score'].max()
    
    if max_score > 0:
        zip_summary['transit_score'] = (zip_summary['raw_score'] / max_score * 10).round(2)
    else:
        zip_summary['transit_score'] = 0
    
    # 確保沒有交通的地方得0分
    zip_summary.loc[zip_summary['raw_score'] == 0, 'transit_score'] = 0
    
    return zip_summary

def main():
    """
    主執行函數
    """
    print("="*60)
    print("MBTA交通分析 - 計算經過每個ZIP的公共交通")
    print("="*60)
    
    # 1. 解壓檔案（如果需要）
    if not os.path.exists('MBTA_data'):
        extract_gtfs_files('MBTA_GTFS.zip')
    
    # 2. 讀取GTFS數據
    stops, routes, stop_times, trips = load_MBTA_data()
    
    # 3. 獲取Boston ZIP codes（使用更寬鬆的邊界）
    boston_zips = get_boston_zips_boundaries()
    
    # 4. 將站點分配到ZIP（允許重疊）
    stop_zip_df = assign_stops_to_multiple_zips(stops, boston_zips)
    
    # 5. 分析經過每個ZIP的路線
    zip_routes = analyze_routes_by_zip(stop_zip_df, stops, routes, stop_times, trips)
    
    # 6. 計算ZIP統計和分數
    zip_transit_summary = calculate_transit_by_zip(zip_routes)
    
    # 7. 加入ZIP名稱
    zip_names = pd.DataFrame.from_dict(
        {k: v['name'] for k, v in boston_zips.items()}, 
        orient='index', 
        columns=['name']
    )
    zip_transit_summary = zip_transit_summary.merge(
        zip_names, 
        left_on='ZIP', 
        right_index=True, 
        how='left'
    )
    
    # 8. 重新排序欄位
    output_columns = [
        'ZIP', 'name', 
        'bus_stops', 'bus_routes',
        'train_stops', 'train_routes',
        'subway_stops', 'subway_routes',
        'transit_score'
    ]
    
    final_output = zip_transit_summary[output_columns].sort_values('ZIP')
    
    # 9. 保存結果
    output_file = 'Boston_transit_scores.csv'
    final_output.to_csv(output_file, index=False)
    print(f"\n結果已保存至: {output_file}")
    
    # 10. 顯示摘要
    print("\n" + "="*60)
    print("Transit Summary by ZIP (Top 10):")
    print("="*60)
    top_10 = final_output.nlargest(10, 'transit_score')
    print(top_10[['ZIP', 'name', 'bus_routes', 'train_routes', 'subway_routes', 'transit_score']].to_string(index=False))
    
    # 檢查South Boston
    print("\n" + "="*60)
    print("South Boston (02127) 檢查:")
    print("="*60)
    sb = final_output[final_output['ZIP'] == '02127']
    if not sb.empty:
        print(sb.to_string(index=False))
    else:
        print("警告：02127沒有找到任何交通數據！")
    
    return final_output

if __name__ == "__main__":
    result = main()

MBTA交通分析 - 計算經過每個ZIP的公共交通

讀取GTFS數據...


<ipython-input-30-9fc86f2aed71>:31: DtypeWarning: Columns (0,3,5) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_csv(f'{folder_path}/stop_times.txt')
<ipython-input-30-9fc86f2aed71>:32: DtypeWarning: Columns (0,4,6) have mixed types. Specify dtype option on import or set low_memory=False.
  trips = pd.read_csv(f'{folder_path}/trips.txt')


- 站點數: 10276
- 路線數: 395
- 班次數: 197164

將站點分配到ZIP codes...
找到 4108 個站點-ZIP配對

分析路線類型...
計算站點-路線關係...

計算ZIP層級統計...

結果已保存至: Boston_transit_scores.csv

Transit Summary by ZIP (Top 10):
  ZIP               name  bus_routes  train_routes  subway_routes  transit_score
02115     Fenway/Kenmore          38             9              1          10.00
02108        Beacon Hill          29             7              3           8.74
02113          North End          37            10              2           8.37
02110 Financial District          29            17              3           7.99
02109           Downtown          26             7              3           7.95
02135           Brighton          27             4              0           7.79
02111          Chinatown          21            16              2           6.68
02114           West End          35            10              2           6.14
02120       Mission Hill          21             5              1           5.91
02116  

# Hospital
hospitals score:  
max = 10, which count=6  
0 means no hospital  

In [40]:
import pandas as pd
import io

# 1. Read the CSV file
# Ensure Zipcode is read as string from the start for consistency
df_overall = pd.read_csv("hospitals.csv", dtype={'Zipcode': str})
df_detail = pd.read_csv("hospitals.csv", dtype={'Zipcode': str})

# 2. Prepare data for Sheet 1: "Overall"
# Group df_overall by Zipcode and count the number of hospitals
# The operation is performed on df_overall, not the undefined df
df_overall = df_overall.groupby('Zipcode').size().reset_index(name='Hospital_Count')
max_score = df_overall["Hospital_Count"].max() # which is 6
df_overall["Hospital_Score"] = (df_overall["Hospital_Count"] / max_score * 10).round(2)

# 3. Prepare data for Sheet 2: "Detail"
# Select relevant columns (Zipcode and Name) from df_detail 
# The operation is performed on df_detail, not the undefined df
df_detail = df_detail[['Zipcode', 'Name']].sort_values(by=['Zipcode', 'Name']).reset_index(drop=True)

# 4. Write both DataFrames to an Excel file with two sheets
output_file = 'Boston_hospital_score.xlsx'
with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    df_overall.to_excel(writer, sheet_name='Overall', index=False)
    df_detail.to_excel(writer, sheet_name='Detail', index=False)

print(f"Successfully created '{output_file}' with two sheets: 'Overall' and 'Detail'.")

# Optional: Display the first few rows of the data for verification
print("\n--- Overall Sheet Data ---")
print(df_overall)

print("\n--- Detail Sheet Data (First 5 Rows) ---")
print(df_detail.head())

Successfully created 'Boston_hospital_score.xlsx' with two sheets: 'Overall' and 'Detail'.

--- Overall Sheet Data ---
   Zipcode  Hospital_Count  Hospital_Score
0    02111               1            1.67
1    02114               6           10.00
2    02115               4            6.67
3    02116               1            1.67
4    02118               3            5.00
5    02119               2            3.33
6    02120               1            1.67
7    02124               1            1.67
8    02129               1            1.67
9    02130               2            3.33
10   02131               1            1.67
11   02132               1            1.67
12   02135               2            3.33
13   02215               3            5.00

--- Detail Sheet Data (First 5 Rows) ---
  Zipcode                                   Name
0   02111                   Tufts Medical Center
1   02114  Erich LIindemann Mental Health Center
2   02114                  Mass General Hospita

# Population Density


In [12]:
df_density = pd.read_excel('population_density.xlsx')
df_density = df_density[(df_density["state_id"] == "MA") & (df_density["county_name"] == "Suffolk")]
print(df_density.head())

# only keep ZIP code, City, Population, Density
df_density = df_density[['zip','city','population','density']]
df_density.to_csv('Boston_popu_density.csv', index=False, encoding='utf-8-sig')

      zip       lat       lng    city state_id     state_name  zcta  \
468  2108  42.35657 -71.06516  Boston       MA  Massachusetts  True   
469  2109  42.36498 -71.05277  Boston       MA  Massachusetts  True   
470  2110  42.35797 -71.05189  Boston       MA  Massachusetts  True   
471  2111  42.35067 -71.06054  Boston       MA  Massachusetts  True   
472  2113  42.36526 -71.05524  Boston       MA  Massachusetts  True   

     parent_zcta  population  density  county_fips county_name  \
468          NaN      4178.0   5890.1        25025     Suffolk   
469          NaN      3714.0   8456.3        25025     Suffolk   
470          NaN      2178.0   4701.0        25025     Suffolk   
471          NaN      8067.0  15890.8        25025     Suffolk   
472          NaN      6522.0  25874.3        25025     Suffolk   

     county_weights county_names_all county_fips_all  imprecise  military  \
468  {"25025": 100}          Suffolk           25025      False     False   
469  {"25025": 100}   

# Housing Price

500000 ~ 1600000  
10 --> 1

In [48]:
import pandas as pd
import numpy as np

# Define constants based on user requirements
X_MIN = 500000    # 對應 10 分
X_MAX = 1600000   # 對應 1 分
RANGE_X = X_MAX - X_MIN # 1,100,000
SCORE_MAX = 10
SCORE_RANGE = SCORE_MAX - 1 # 10

# Load the data
df_house = pd.read_excel("boston_median_housing_price.xlsx")

# 確保 'home_value' 是數字
df_house['home_value'] = pd.to_numeric(df_house['home_value'], errors='coerce')

# --- 修正後的線性反比分數計算 ---
# S = S_MAX - ((X - X_MIN) / RANGE_X) * SCORE_RANGE
df_house["housing_price_score"] = (
    SCORE_MAX - ((df_house["home_value"] - X_MIN) / RANGE_X) * SCORE_RANGE
).round(2)

# 處理超出範圍的值：分數必須介於 1 到 10 之間
df_house["housing_price_score"] = df_house["housing_price_score"].clip(lower=1, upper=10)

df_house
# Export to CSV
output_file = 'Boston_median_housing_price_score.csv'
df_house.to_csv(output_file, index=False, encoding='utf-8-sig')

# Combine to a one file

In [5]:
import pandas as pd
import numpy as np

# --- 1. 定義檔案名稱及欄位名稱映射 ---

file_map = {
    # 順序: Safety (CSV)
    "safety": {
        "file": "Boston_safety_score.csv",
        "zip_col": "ZIP_code",
        "prefix": "Safety"
    },
    # 順序: Housing (CSV)
    "housing": {
        "file": "Boston_median_housing_price_score.csv",
        "zip_col": "ZIP code",
        "prefix": "Housing"
    },
    # 順序: Education (CSV)
    "education": {
        "file": "Boston_school_scores.csv",
        "zip_col": "ZIP",
        "prefix": "Education"
    },
    # 順序: Healthcare (xlsx) - 讀取時將使用 pd.read_excel
    "healthcare": {
        "file": "Boston_hospital_score.xlsx", 
        "zip_col": "Zipcode",
        "prefix": "Healthcare",
        "sheet_name": "Overall" # Excel 工作表名稱
    },
    # 順序: Transit (CSV)
    "transit": {
        "file": "Boston_transit_scores.csv",
        "zip_col": "ZIP",
        "prefix": "Transit"
    }
}

# --- 2. 載入並標準化各個數據表 ---

dfs = {}
for key, mapping in file_map.items():
    filename = mapping["file"]
    
    if filename.endswith(('.xlsx', '.xls')):
        # 讀取 Excel 檔案
        df = pd.read_excel(filename, sheet_name=mapping.get("sheet_name", 0))
    else:
        # 讀取 CSV 檔案
        try:
            df = pd.read_csv(filename, encoding='utf-8-sig')
        except UnicodeDecodeError:
            df = pd.read_csv(filename, encoding='latin-1')
        
    df = df.rename(columns={mapping["zip_col"]: "ZIP_code"})
    
    new_columns = {col: f"{mapping['prefix']}_{col}" 
                   for col in df.columns if col != "ZIP_code"}
    df = df.rename(columns=new_columns)
    
    dfs[key] = df

# --- 3. 執行 Full Outer Join 整合數據 ---

merge_order_keys = ['safety', 'housing', 'education', 'healthcare', 'transit']
df_merged = dfs[merge_order_keys[0]].copy() 

for key in merge_order_keys[1:]:
    df_merged = pd.merge(
        df_merged, 
        dfs[key], 
        on='ZIP_code', 
        how='outer'
    )

# 統一 ZIP_code 格式為 5 位字串 (例如: 2111 -> 02111)
df_merged['ZIP_code'] = df_merged['ZIP_code'].astype(str).str.zfill(5)


# --- 4. 篩選 ZIP code (移除特定 ZIP code) ---
zip_codes_to_remove = ['02150', '02151', '02152', '02467']
df_merged = df_merged[~df_merged['ZIP_code'].isin(zip_codes_to_remove)].copy()


# --- 5. 針對 Score 欄位補 0 (其餘欄位保持 NaN) ---
score_columns = [col for col in df_merged.columns if '_score' in col.lower()]
df_merged[score_columns] = df_merged[score_columns].fillna(0)


# --- 6. 重新排列欄位順序 ---

final_columns = ['ZIP_code']
current_cols = df_merged.columns.tolist()

for key in merge_order_keys:
    prefix = file_map[key]['prefix']
    group_cols = [col for col in current_cols if col.startswith(f"{prefix}_")]
    final_columns.extend(group_cols)

df_final = df_merged[final_columns]

# --- 7. 匯出結果 ---

output_file = 'Boston_Data_All_Final.csv'
df_final.to_csv(output_file, index=False, encoding='utf-8-sig')

# 輸出結果摘要
print(f"\n--- 成功! (Success!) ---")
print(f"五張表已整合，並已移除 ZIP code: {zip_codes_to_remove}。")
print(f"所有分數欄位 (含 '_score' 的欄位) 的缺失值已補 0。檔案已儲存至: '{output_file}'.")
print("\n--- 整合結果 (前 5 筆) ---")
print(f"總欄位數: {len(df_final.columns)}")
print(df_final.head().to_markdown(index=False))


--- 成功! (Success!) ---
五張表已整合，並已移除 ZIP code: ['02150', '02151', '02152', '02467']。
所有分數欄位 (含 '_score' 的欄位) 的缺失值已補 0。檔案已儲存至: 'Boston_Data_All_Final.csv'.

--- 整合結果 (前 5 筆) ---
總欄位數: 25
|   ZIP_code |   Safety_Incident_count |   Safety_Shooting_count |   Safety_Unsafe_Score |   Safety_Safety_score |   Housing_home_value |   Housing_housing_price_score |   Education_School_Elementary |   Education_School_Middle |   Education_School_High |   Education_School_Other |   Education_Total_Schools | Education_Has_Complete_K12   |   Education_Raw_Education_Score |   Education_Education_Score_10 |   Healthcare_Hospital_Count |   Healthcare_Hospital_Score | Transit_name       |   Transit_bus_stops |   Transit_bus_routes |   Transit_train_stops |   Transit_train_routes |   Transit_subway_stops |   Transit_subway_routes |   Transit_transit_score |
|-----------:|------------------------:|------------------------:|----------------------:|----------------------:|---------------------:|-----------------

### overall score

In [6]:
import pandas as pd
import numpy as np

# 讀取現有的數據
df = pd.read_csv('Boston_Data_All_Final.csv')

print("="*60)
print("計算Boston ZIP Code Overall Score")
print("="*60)

# 顯示原始欄位
print("\n原始數據欄位:")
print(df.columns.tolist())

# 確認各項評分欄位名稱並顯示範圍
print("\n各項評分範圍:")
print(f"Safety Score: {df['Safety_Safety_score'].min():.2f} - {df['Safety_Safety_score'].max():.2f}")
print(f"Housing Score: {df['Housing_housing_price_score'].min():.2f} - {df['Housing_housing_price_score'].max():.2f}")
print(f"Education Score: {df['Education_Education_Score_10'].min():.2f} - {df['Education_Education_Score_10'].max():.2f}")
print(f"Healthcare Score: {df['Healthcare_Hospital_Score'].min():.2f} - {df['Healthcare_Hospital_Score'].max():.2f}")
print(f"Transit Score: {df['Transit_transit_score'].min():.2f} - {df['Transit_transit_score'].max():.2f}")

# 計算Overall Score
# 權重：safety 35%, housing 25%, school 15%, hospital 15%, transport 10%
weights = {
    'safety': 0.35,
    'housing': 0.25,
    'education': 0.15,
    'healthcare': 0.15,
    'transit': 0.10
}

print("\n計算Overall Score...")
print(f"權重配置:")
print(f"  Safety: {weights['safety']*100}%")
print(f"  Housing: {weights['housing']*100}%")
print(f"  Education_Education_Score_10: {weights['education']*100}%")
print(f"  Healthcare_Hospital_Score: {weights['healthcare']*100}%")
print(f"  Transit_transit_score: {weights['transit']*100}%")

# 計算加權總分
df['Overall_Score'] = (
    df['Safety_Safety_score'] * weights['safety'] +
    df['Housing_housing_price_score'] * weights['housing'] +
    df['Education_Education_Score_10'] * weights['education'] +
    df['Healthcare_Hospital_Score'] * weights['healthcare'] +
    df['Transit_transit_score'] * weights['transit']
)

# 四捨五入到小數點後兩位
df['Overall_Score'] = df['Overall_Score'].round(2)

# 確保Overall Score在0-10範圍內
df['Overall_Score'] = df['Overall_Score'].clip(0, 10)

# 排序（按Overall Score從高到低）
df_sorted = df.sort_values('Overall_Score', ascending=False)

# 儲存更新後的檔案
df_sorted.to_csv('Boston_Data_All_Final.csv', index=False)
print("\n檔案已更新並儲存: Boston_Data_All_Final.csv")

# 顯示Top 10最佳ZIP區域
print("\n" + "="*60)
print("Top 10 最佳居住ZIP區域 (Overall Score):")
print("="*60)

top10 = df_sorted.head(10)[['ZIP_code', 'Overall_Score', 
                             'Safety_Safety_score', 'Housing_housing_price_score',
                             'Education_Education_Score_10', 'Healthcare_Hospital_Score',
                             'Transit_transit_score']]

# 重命名欄位以便顯示
top10_display = top10.copy()
top10_display.columns = ['ZIP', 'Overall', 'Safety', 'Housing', 'Education', 'Healthcare', 'Transit']
print(top10_display.to_string(index=False))

# 顯示整體統計
print("\n" + "="*60)
print("Overall Score 統計:")
print("="*60)
print(f"最高分: {df['Overall_Score'].max():.2f}")
print(f"最低分: {df['Overall_Score'].min():.2f}")
print(f"平均分: {df['Overall_Score'].mean():.2f}")
print(f"中位數: {df['Overall_Score'].median():.2f}")

# 分數分布
print("\n分數分布:")
print(f"8-10分 (極佳): {len(df[df['Overall_Score'] >= 8])} 個ZIP")
print(f"6-8分 (良好): {len(df[(df['Overall_Score'] >= 6) & (df['Overall_Score'] < 8)])} 個ZIP")
print(f"4-6分 (中等): {len(df[(df['Overall_Score'] >= 4) & (df['Overall_Score'] < 6)])} 個ZIP")
print(f"2-4分 (較差): {len(df[(df['Overall_Score'] >= 2) & (df['Overall_Score'] < 4)])} 個ZIP")
print(f"0-2分 (很差): {len(df[df['Overall_Score'] < 2])} 個ZIP")

print("\n完成！Overall_Score已新增到CSV檔案最後一欄。")

計算Boston ZIP Code Overall Score

原始數據欄位:
['ZIP_code', 'Safety_Incident_count', 'Safety_Shooting_count', 'Safety_Unsafe_Score', 'Safety_Safety_score', 'Housing_home_value', 'Housing_housing_price_score', 'Education_School_Elementary', 'Education_School_Middle', 'Education_School_High', 'Education_School_Other', 'Education_Total_Schools', 'Education_Has_Complete_K12', 'Education_Raw_Education_Score', 'Education_Education_Score_10', 'Healthcare_Hospital_Count', 'Healthcare_Hospital_Score', 'Transit_name', 'Transit_bus_stops', 'Transit_bus_routes', 'Transit_train_stops', 'Transit_train_routes', 'Transit_subway_stops', 'Transit_subway_routes', 'Transit_transit_score']

各項評分範圍:
Safety Score: 0.00 - 10.00
Housing Score: 1.15 - 9.86
Education Score: 0.00 - 10.00
Healthcare Score: 0.00 - 10.00
Transit Score: 0.00 - 10.00

計算Overall Score...
權重配置:
  Safety: 35.0%
  Housing: 25.0%
  Education_Education_Score_10: 15.0%
  Healthcare_Hospital_Score: 15.0%
  Transit_transit_score: 10.0%

檔案已更新並儲存: Bo